# 06 — Final Model Evaluation & Comparison

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Load all saved models

In [2]:
from src.models.xgboost_model import XGBForecaster
from src.features import get_feature_columns

featured = pd.read_csv('../data/processed/features.csv', parse_dates=['date'])
FEATURE_COLS = [c for c in get_feature_columns() if c in featured.columns]
TARGET = 'new_cases_7d'

cutoff = featured['date'].max() - pd.Timedelta(days=30)
test = featured[featured.date > cutoff]
X_test = test[FEATURE_COLS].fillna(0).values
y_test = test[TARGET].fillna(0).values

xgb = XGBForecaster.load('../models/saved/xgb.pkl')
preds_xgb = xgb.predict(X_test)

## 2. Evaluate all models

In [3]:
from src.evaluation import evaluate, compare_models

results = [
    evaluate(y_test, preds_xgb, 'XGBoost'),
    # Add LSTM and SIR results here after running notebooks 3 & 5
]
summary = compare_models(results)
summary

  [XGBoost]  RMSE=173.3  MAE=29.9  MAPE=4.5%  R²=0.9976

── Model Comparison ──────────────────────────────
          RMSE   MAE  MAPE   R2
model                          
XGBoost 173.26 29.94  4.49 1.00
──────────────────────────────────────────────────


,RMSE,MAE,MAPE,R2
model,,,,
XGBoost,173.258358,29.94091,4.48918,0.99756


## 3. Actual vs predicted — 5 countries

In [4]:
countries = test['country'].unique()[:5]
fig, axes = plt.subplots(len(countries), 1, figsize=(14, 4*len(countries)))
fig.patch.set_facecolor('#050a0f')

for ax, c in zip(axes, countries):
    sub = test[test.country == c].reset_index(drop=True)
    Xi = sub[FEATURE_COLS].fillna(0).values
    pi = xgb.predict(Xi)
    yi = sub[TARGET].fillna(0).values
    ax.set_facecolor('#0a1520')
    ax.plot(sub['date'], yi, color='#7a9e90', label='Actual', linewidth=1.2)
    ax.plot(sub['date'], pi, color='#00c896', label='XGB Predicted', linewidth=1.6, linestyle='--')
    ax.set_title(c, color='#d0e8e0', fontsize=11)
    ax.legend(facecolor='#0d1b2a', edgecolor='#1a3040', labelcolor='#7a9e90', fontsize=9)
    ax.grid(True, color='#0f1e2e', linewidth=0.4)
    ax.tick_params(colors='#3a5e50')

plt.tight_layout()
plt.savefig('../outputs/plots/model_comparison.png', dpi=120,
            bbox_inches='tight', facecolor='#050a0f')
plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_18612\3863497935.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Error distribution

In [5]:
errors = y_test - preds_xgb
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#050a0f')
for ax in (ax1, ax2):
    ax.set_facecolor('#0a1520')
    ax.tick_params(colors='#3a5e50')

ax1.hist(errors, bins=60, color='#00c896', alpha=0.7, edgecolor='none')
ax1.set_title('Residual Distribution', color='#d0e8e0')
ax1.axvline(0, color='#ffb830', linestyle='--')

ax2.scatter(preds_xgb[::10], errors[::10], alpha=0.3, s=8, color='#00c896')
ax2.axhline(0, color='#ffb830', linestyle='--')
ax2.set_title('Residuals vs Predicted', color='#d0e8e0')
ax2.set_xlabel('Predicted', color='#7a9e90')
ax2.set_ylabel('Error', color='#7a9e90')

plt.tight_layout()
plt.savefig('../outputs/plots/residuals.png', dpi=120,
            bbox_inches='tight', facecolor='#050a0f')
plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_18612\339380506.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
